<div class="alert alert-block alert-success">

#### **DATA MINING PROJECT TEST FILE**

### Importing the libraries:

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix , accuracy_score, f1_score, classification_report

__________________________________
### Loading all the saved files:

In [ ]:
final_model = joblib.load('final_model_voting_rf3_xgb1.pkl')
scaler = joblib.load('scaler.pkl')
with open('final_features.txt', 'r') as f:
    final_features = [line.strip() for line in f.readlines()]

imputer_f16 = joblib.load('imputer_f16.pkl')
imputer_f20 = joblib.load('imputer_f20.pkl')
winsor_limits = joblib.load('winsor_limits.pkl')
encoder = joblib.load('onehot_encoder.pkl')

________________
### Preprocessing Function:

In [ ]:
def preprocess_test_data(df_raw, scaler, final_features,
                         imputer_f16=None, imputer_f20=None,
                         winsor_limits=None, log_scaler=None, encoder=None):
    """
    Applies the same preprocessing steps as in the training script.
    """
    X = df_raw.copy()

    # 1. Drop f12 and f6 early
    X = X.drop(columns=['f12', 'f6'], errors='ignore')

    # 2. Impute f16 and f20 with median
    for col, imputer in [('f16', imputer_f16), ('f20', imputer_f20)]:
        if col in X.columns and X[col].isnull().any():
            X[col] = imputer.transform(X[[col]])

    # 3. Winsorize f16 and f20 
    def winsorize_series(series, low, high):
        return series.clip(lower=low, upper=high)

    for col in ['f16', 'f20']:
        if col in X.columns:
            low = winsor_limits[f'{col}_low']
            high = winsor_limits[f'{col}_high']

    # 4. Drop rows with any remaining NaN (as in your training code)
    X = X.dropna()

    # 5. Extract time features from f15
    if 'f15' in X.columns:
        f15_dt = pd.to_datetime(X['f15'], errors='coerce')
        X['hour'] = f15_dt.dt.hour
        X['dayofweek'] = f15_dt.dt.dayofweek
        X['month'] = f15_dt.dt.month
        X['is_weekend'] = (X['dayofweek'] >= 5).astype(int)

    # 6. Compute duration from f2 and f3
    if 'f2' in X.columns and 'f3' in X.columns:
        X['f2'] = X['f2'].apply(lambda x: x.split('.')[0])
        X['f3'] = X['f3'].apply(lambda x: x.split('.')[0])
        f2_dt = pd.to_datetime(X['f2'], errors='coerce')
        f3_dt = pd.to_datetime(X['f3'], errors='coerce')
        duration_min = (f3_dt - f2_dt).dt.total_seconds() / 60.0
        X['duration_minutes'] = duration_min

    # Drop original date columns
    X = X.drop(columns=['f15', 'f2', 'f3'], errors='ignore')

    # 7. Split f13 (take part after '/')
    if 'f13' in X.columns:
        X['f13'] = X['f13'].apply(lambda x: x.split('/')[1] if pd.notnull(x) else x)

    # 8. Log transform on f7: log(f7 + shift)
    shift = 1e-15
    if 'f7' in X.columns:
        X['f7'] = np.log(X[['f7']] + shift)

    # 9. Feature engineering: multi-hot coding for f21 (weather)
    if 'f21' in X.columns:
        X['Weather_Clear'] = X['f21'].str.contains('clear|fair|sunny', case=False, na=False).astype(int)
        X['Weather_Cloudy'] = X['f21'].str.contains('cloud|overcast|mostly cloudy|partly cloudy|scattered clouds', case=False, na=False).astype(int)
        X['Weather_Rain'] = X['f21'].str.contains('rain|drizzle|shower|thunderstorm', case=False, na=False).astype(int)
        X['Weather_Snow'] = X['f21'].str.contains('snow|blizzard|flurry|ice pellets', case=False, na=False).astype(int)
        X['Weather_Fog'] = X['f21'].str.contains('fog|mist|haze', case=False, na=False).astype(int)
        weather_cols = ['Weather_Clear','Weather_Cloudy','Weather_Rain','Weather_Snow','Weather_Fog']
        X['Weather_Other'] = (X[weather_cols].sum(axis=1) == 0).astype(int)
        X = X.drop(columns=['f21'])

    # 10. Feature engineering: multi-hot coding for f19 (wind direction)
    if 'f19' in X.columns:
        X['Wind_Calm'] = X['f19'].str.contains('calm', case=False, na=False).astype(int)
        X['Wind_Variable'] = X['f19'].str.contains('var|variable', case=False, na=False).astype(int)
        X['Wind_North'] = X['f19'].str.contains('north|nne|nnw|ne|nw', case=False, na=False).astype(int)
        X['Wind_South'] = X['f19'].str.contains('south|sse|ssw|se|sw', case=False, na=False).astype(int)
        X['Wind_East'] = X['f19'].str.contains('east|ene|ese', case=False, na=False).astype(int)
        X['Wind_West'] = X['f19'].str.contains('west|wnw|wsw', case=False, na=False).astype(int)
        # handle single-letter codes
        f19_clean = X['f19'].str.strip()
        X.loc[f19_clean == 'N', 'Wind_North'] = 1
        X.loc[f19_clean == 'S', 'Wind_South'] = 1
        X.loc[f19_clean == 'E', 'Wind_East'] = 1
        X.loc[f19_clean == 'W', 'Wind_West'] = 1
        X = X.drop(columns=['f19'])

    # 11. Convert boolean f22–f31 to 0/1
    bool_cols = [f'f{i}' for i in range(22, 32)]
    for col in bool_cols:
        if col in X.columns:
            X[col] = X[col].astype(int)

    # 12. Create Day/Night indicators from f33, f34
    if 'f33' in X.columns:
        X['Day/Night1'] = (X['f33'] == 'Day').astype(int)
    if 'f34' in X.columns:
        X['Day/Night2'] = (X['f34'] == 'Day').astype(int)
    X = X.drop(columns=['f33', 'f34'], errors='ignore')

    # 13. Drop columns that were removed during training
    X = X.drop(columns=['f32', 'f18', 'f8', 'f9', 'f10', 'f11', 'f14'], errors='ignore')

    # 14. One-hot encode f1 and f13
    cat_cols = ['f1', 'f13']
    existing_cat = [c for c in cat_cols if c in X.columns]
    if existing_cat:
        # Use the fitted encoder
        encoded = encoder.transform(X[existing_cat])
        enc_df = pd.DataFrame(encoded,
                              columns=encoder.get_feature_names_out(existing_cat),
                              index=X.index)
        X = pd.concat([X.drop(columns=existing_cat), enc_df], axis=1)
        

    # 15. Round numeric columns as in training
    numerical_cols = ['f4','f5','f7','f16','f17','f18','f20','duration_minutes']
    for col in numerical_cols:
        if col in X.columns:
            X[col] = np.round(X[col], 3)
    if 'duration_minutes' in X.columns:
        X['duration_minutes'] = np.round(X['duration_minutes'], 2)

    # 16. Reindex to final_features (add missing columns as 0)
    missing = set(final_features) - set(X.columns)
    if missing:
        print(f"Info: adding missing columns: {missing}")
        for col in missing:
            X[col] = 0
    X = X[final_features]

    # 17. Standardize continuous features with the saved StandardScaler
    continuous_features = ['f4','f5','f7','f16','f17','f18','f20','duration_minutes']
    to_scale = [c for c in continuous_features if c in X.columns]
    if to_scale:
        X[to_scale] = scaler.transform(X[to_scale])

    return X

______________
### Read Test File:

In [ ]:
# مسیر فایل تست را وارد کنید
df_test = pd.read_csv('test.csv')
df_test.shape

In [ ]:
y_test = df_test['y'].copy()
X_test = df_test.drop(columns=['y'])

_______________
### Preprocessing:

In [ ]:
X_test_processed = preprocess_test_data(
    X_test,
    scaler,
    final_features,
    imputer_f16=imputer_f16,
    imputer_f20=imputer_f20,
    winsor_limits=winsor_limits,
    encoder=encoder
)

y_test = y_test.loc[X_test_processed.index]

X_test_processed.shape

____________________
### Predict and Save:


In [ ]:
y_pred = final_model.predict(X_test_processed)

output = pd.DataFrame({'predictions': y_pred, 'real target': y_test})
output.to_csv('predictions.csv', index=False)
print("Predictions saved to 'predictions.csv'")

_______________
### Evaluation:

In [ ]:
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-score (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1-score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")

In [ ]:
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=sorted(y_test.unique()),
            yticklabels=sorted(y_test.unique()))
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show